# Assemble a filtered microbiology ontology

Download selected OBO Foundry resources, retain terms relevant to pathogen/host/transmission literature, and write a compact ontology catalog. The existing local application ontology is kept as the prompt-facing schema; downloaded terms and cross-references are recorded alongside it for traceable enrichment.

Run this notebook before `03_semantic_abstraction.ipynb`. Adjust the regular expressions in `RESOURCE_SPECS` when the corpus or research question changes.

In [ ]:
from collections import defaultdict
from pathlib import Path
from urllib.request import Request, urlopen
import hashlib
import json
import re
import xml.etree.ElementTree as ET

import yaml

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'assets').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'assets').exists():
    raise FileNotFoundError('Could not locate the repository assets directory.')

ONTOLOGY_ROOT = PROJECT_ROOT / 'assets' / 'ontologies'
RAW_ROOT = ONTOLOGY_ROOT / 'external'
BASE_ONTOLOGY_PATH = ONTOLOGY_ROOT / 'entity_ontology_microbiology.yaml'
ASSEMBLED_ONTOLOGY_PATH = ONTOLOGY_ROOT / 'entity_ontology_microbiology_assembled.yaml'
TERM_CATALOG_PATH = ONTOLOGY_ROOT / 'microbiology_external_terms.json'

## Select resources and vocabulary

These defaults cover infectious disease, pathogen–host phenotypes, environments, genomic epidemiology, and microbial phenotypes. The complete NCBITaxon OBO release is intentionally optional because it is much larger than the domain ontologies; enable it only when taxonomy terms are needed in the local catalog.

In [ ]:
RESOURCE_SPECS = {
    'ido': {
        'enabled': True,
        'url': 'https://raw.githubusercontent.com/infectious-disease-ontology/infectious-disease-ontology/master/ido.owl',
        'format': 'owl',
        'patterns': [
            r'\binfectious disease\b', r'\binfection\b', r'\bpathogen\b',
            r'\bhost\b', r'\btransmission\b', r'\bexposure\b',
            r'\boutbreak\b', r'\bsurveillance\b', r'\breservoir\b',
        ],
    },
    'phipo': {
        'enabled': True,
        'url': 'https://purl.obolibrary.org/obo/phipo.obo',
        'format': 'obo',
        'patterns': [
            r'\bpathogen\b', r'\bhost\b', r'\bvirulence\b',
            r'\battenuat', r'\binfection\b', r'\bsusceptib',
        ],
    },
    'envo': {
        'enabled': True,
        'url': 'https://purl.obolibrary.org/obo/envo.obo',
        'format': 'obo',
        'patterns': [
            r'\benvironment', r'\bhabitat\b', r'\bwater\b', r'\bsoil\b',
            r'\bfood\b', r'\bplant\b', r'\bsap\b', r'\bsurface\b',
            r'\breservoir\b', r'\bsample\b',
        ],
    },
    'genepio': {
        'enabled': True,
        'url': 'https://raw.githubusercontent.com/GenEpiO/genepio/master/genepio.obo',
        'format': 'obo',
        'patterns': [
            r'\boutbreak\b', r'\bcase\b', r'\bspecimen\b', r'\bisolate\b',
            r'\bpathogen\b', r'\btransmission\b', r'\bfood\b',
            r'\bgenom', r'\bsequence\b', r'\bassay\b',
        ],
    },
    'omp': {
        'enabled': True,
        'url': 'https://purl.obolibrary.org/obo/omp.obo',
        'format': 'obo',
        'patterns': [
            r'\bvirulence\b', r'\bpathogen\b', r'\bhost\b',
            r'\bgrowth\b', r'\btemperature\b', r'\bpH\b', r'\bsalinity\b',
            r'\boxygen\b', r'\bresistance\b', r'\bmorphology\b',
            r'\binfection\b', r'\bbiofilm\b', r'\bmotility\b',
        ],
    },
    'ncbitaxon': {
        'enabled': False,
        'url': 'https://purl.obolibrary.org/obo/ncbitaxon.obo',
        'format': 'obo',
        'patterns': [r'\bnipah virus\b', r'\bpteropus\b', r'\bhomo sapiens\b', r'\bparamyxoviridae\b'],
    },
}

USER_AGENT = 'RecursiveFraming ontology assembler/0.1'

PROMPT_REFERENCE_PATTERNS = {
    'pathogen': [r'\bpathogen\b', r'\binfectious agent\b', r'\bvirus\b'],
    'host': [r'\bhost\b', r'\borganism\b'],
    'reservoir_host': [r'\breservoir\b', r'\bmaintenance host\b'],
    'exposure': [r'\bexposure\b', r'\bcontact event\b'],
    'transmission_route': [r'\btransmission\b', r'\broute\b'],
    'outbreak': [r'\boutbreak\b', r'\bcase cluster\b'],
    'specimen': [r'\bspecimen\b', r'\bsample\b'],
    'assay': [r'\bassay\b', r'\blaboratory test\b'],
    'genomic_sequence': [r'\bsequence\b', r'\bgenome\b'],
    'geographic_location': [r'\bgeographic\b', r'\blocation\b', r'\bregion\b'],
}

In [ ]:
def download_resource(name, spec):
    destination = RAW_ROOT / f"{name}.{spec.get('format', 'obo')}"
    if destination.exists():
        return destination
    request = Request(spec['url'], headers={'User-Agent': USER_AGENT})
    with urlopen(request, timeout=120) as response:
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(response.read())
    return destination


def parse_obo(path):
    terms = {}
    current = None
    in_term = False
    for raw_line in path.read_text(encoding='utf-8', errors='replace').splitlines():
        line = raw_line.strip()
        if line == '[Term]':
            current = {}
            in_term = True
            continue
        if line.startswith('['):
            current = None
            in_term = False
            continue
        if not line or line.startswith('!'):
            continue
        if current is None or not in_term:
            continue
        key, separator, value = line.partition(': ')
        if not separator:
            continue
        if key in {'id', 'name', 'def', 'is_a'}:
            current[key] = value
        elif key == 'synonym':
            current.setdefault('synonym', []).append(value)
        elif key == 'is_obsolete' and value == 'true':
            current['is_obsolete'] = True
        if key == 'id':
            terms[value] = current
    return terms


def parse_owl(path):
    namespaces = {
        'rdf': 'http://www.w3.org/1999/02/22-rdf-syntax-ns#',
        'rdfs': 'http://www.w3.org/2000/01/rdf-schema#',
    }
    root = ET.fromstring(path.read_bytes())
    terms = {}
    for element in root.iter():
        if not element.tag.endswith('Class'):
            continue
        iri = element.attrib.get(f"{{{namespaces['rdf']}}}about") or element.attrib.get(f"{{{namespaces['rdf']}}}ID")
        if not iri:
            continue
        term_id = iri.rsplit('/obo/', 1)[-1].rsplit('#', 1)[-1]
        term = {'id': term_id, 'synonym': []}
        for child in element:
            local_name = child.tag.rsplit('}', 1)[-1]
            value = (child.text or '').strip()
            if local_name == 'subClassOf':
                parent_iri = child.attrib.get(f"{{{namespaces['rdf']}}}resource")
                if parent_iri:
                    term['is_a'] = parent_iri.rsplit('/obo/', 1)[-1].rsplit('#', 1)[-1]
            elif local_name == 'label' and value:
                term['name'] = value
            elif local_name in {'IAO_0000115', 'definition'} and value:
                term['def'] = value
            elif 'Synonym' in local_name and value:
                term['synonym'].append(value)
            elif local_name in {'deprecated', 'is_obsolete'} and value.lower() == 'true':
                term['is_obsolete'] = True
        if term.get('name'):
            terms[term_id] = term
    return terms


def parse_resource(path, spec):
    return parse_owl(path) if spec.get('format') == 'owl' else parse_obo(path)


def clean_quoted(value):
    match = re.match(r'\"(.*?)\"', value)
    return match.group(1) if match else value


def ancestors(term_id, terms):
    result = set()
    pending = [term_id]
    while pending:
        current_id = pending.pop()
        parent_line = terms.get(current_id, {}).get('is_a')
        if not parent_line:
            continue
        parent_id = parent_line.split(' ! ', 1)[0].strip()
        if parent_id not in result:
            result.add(parent_id)
            pending.append(parent_id)
    return result


def filter_terms(name, terms, patterns):
    expressions = [re.compile(pattern, re.IGNORECASE) for pattern in patterns]
    matched = set()
    for term_id, term in terms.items():
        if term.get('is_obsolete') or 'name' not in term:
            continue
        text = ' '.join([term.get('name', ''), *term.get('synonym', [])])
        if any(expression.search(text) for expression in expressions):
            matched.add(term_id)
    selected = matched | {parent for term_id in matched for parent in ancestors(term_id, terms)}
    return {term_id: terms[term_id] for term_id in selected if term_id in terms}


In [ ]:
selected_terms = {}
source_metadata = {}
for name, spec in RESOURCE_SPECS.items():
    if not spec['enabled']:
        continue
    raw_path = download_resource(name, spec)
    raw_bytes = raw_path.read_bytes()
    terms = parse_resource(raw_path, spec)
    filtered = filter_terms(name, terms, spec['patterns'])
    selected_terms[name] = filtered
    source_metadata[name] = {
        'url': spec['url'],
        'path': str(raw_path.relative_to(PROJECT_ROOT)),
        'sha256': hashlib.sha256(raw_bytes).hexdigest(),
        'term_count': len(filtered),
    }
    print(f'{name}: {len(terms):,} downloaded terms -> {len(filtered):,} selected terms')


## Build the assembled catalog

The generated YAML remains compatible with `Ontology.from_yaml`: its existing entity and relation types are preserved. The selected external terms are stored in `external_terms` for normalization, auditing, and later ontology-aware extensions.

In [ ]:
def compact_term(source, term_id, term):
    definition = clean_quoted(term.get('def', ''))
    synonyms = [clean_quoted(value) for value in term.get('synonym', [])]
    parent = term.get('is_a', '').split(' ! ', 1)[0].strip() or None
    return {
        'id': term_id,
        'source': source,
        'label': term.get('name'),
        'definition': definition,
        'synonyms': synonyms,
        'parent': parent,
    }


base = yaml.safe_load(BASE_ONTOLOGY_PATH.read_text(encoding='utf-8'))
catalog = {
    source: {
        term_id: compact_term(source, term_id, term)
        for term_id, term in terms.items()
    }
    for source, terms in selected_terms.items()
}

def reference_terms_for(entity_key, limit=8):
    expressions = [re.compile(pattern, re.IGNORECASE) for pattern in PROMPT_REFERENCE_PATTERNS.get(entity_key, [])]
    references = []
    for source, terms in catalog.items():
        for term_id, term in terms.items():
            text = ' '.join([term.get('label', ''), *term.get('synonyms', [])])
            if any(expression.search(text) for expression in expressions):
                references.append({'id': f'{source}:{term_id}', 'label': term['label']})
    return sorted(references, key=lambda item: (item['label'].lower(), item['id']))[:limit]

for entity_key, entity_type in base.get('entity_types', {}).items():
    references = reference_terms_for(entity_key)
    if references:
        entity_type['reference_terms'] = references

base['ontology']['id'] = 'microbiology-zoonosis-assembled'
base['ontology']['version'] = '0.2.0'
base['ontology']['assembly'] = {
    'method': 'download OBO resources, match configured labels/synonyms, and retain is_a ancestors',
    'sources': source_metadata,
}
base['external_terms'] = catalog

ASSEMBLED_ONTOLOGY_PATH.parent.mkdir(parents=True, exist_ok=True)
ASSEMBLED_ONTOLOGY_PATH.write_text(
    yaml.safe_dump(base, sort_keys=False, allow_unicode=True),
    encoding='utf-8',
)
TERM_CATALOG_PATH.write_text(
    json.dumps({'sources': source_metadata, 'terms': catalog}, indent=2, ensure_ascii=False) + '\n',
    encoding='utf-8',
)

print(f'Wrote {ASSEMBLED_ONTOLOGY_PATH.relative_to(PROJECT_ROOT)}')
print(f'Wrote {TERM_CATALOG_PATH.relative_to(PROJECT_ROOT)}')